In [1]:

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
import kagglehub


In [2]:
import torch
import torch.nn as nn
batch_size=32
seq_len =10
input_dim =1

def generate_synthetic_sequence_data(batch_size, seq_len, input_dim):
    X = torch.randn(batch_size, seq_len, input_dim)
    s = X.sum(dim=1)
    y = (s > 0 ).long()
    y= y.squeeze(1)
    return X,y

X, y = generate_synthetic_sequence_data(32, 10, 1)
print("X.shape:", X.shape)
print("y.shape:", y.shape)

X.shape: torch.Size([32, 10, 1])
y.shape: torch.Size([32])


In [3]:
class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size , num_layers, num_classes=1):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)
    def forward(self,x):
        output, (h_n, C_n) =self.lstm(x)
        last_hidden = h_n[-1]
        logits = self.fc(last_hidden)
        return logits

In [4]:
model =  LSTMClassifier(
    input_size= 1,
    hidden_size = 16,
    num_layers=1,
    num_classes =1 
)

In [5]:
logits = model(X)
print("logits.shape:", logits.shape)

logits.shape: torch.Size([32, 1])


In [6]:

train_size = 1000
X_train, y_train = generate_synthetic_sequence_data(train_size, 10, 1)
test_size = 200
X_test, y_test = generate_synthetic_sequence_data(test_size, 10, 1)

In [7]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 1e-3)

for epochs in range(100):
    X, y = generate_synthetic_sequence_data(32, 10, 1)
    logits = model(X_train)
    loss = criterion(logits.squeeze(1),y_train.float())
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if (epochs+1) % 2==0:
        print(f"Epochs {epochs+1}, loss: {loss.item(): .4f} ")

Epochs 2, loss:  0.6918 
Epochs 4, loss:  0.6908 
Epochs 6, loss:  0.6899 
Epochs 8, loss:  0.6890 
Epochs 10, loss:  0.6881 
Epochs 12, loss:  0.6871 
Epochs 14, loss:  0.6862 
Epochs 16, loss:  0.6853 
Epochs 18, loss:  0.6843 
Epochs 20, loss:  0.6833 
Epochs 22, loss:  0.6823 
Epochs 24, loss:  0.6813 
Epochs 26, loss:  0.6802 
Epochs 28, loss:  0.6790 
Epochs 30, loss:  0.6778 
Epochs 32, loss:  0.6766 
Epochs 34, loss:  0.6752 
Epochs 36, loss:  0.6737 
Epochs 38, loss:  0.6722 
Epochs 40, loss:  0.6705 
Epochs 42, loss:  0.6686 
Epochs 44, loss:  0.6665 
Epochs 46, loss:  0.6643 
Epochs 48, loss:  0.6617 
Epochs 50, loss:  0.6589 
Epochs 52, loss:  0.6556 
Epochs 54, loss:  0.6520 
Epochs 56, loss:  0.6478 
Epochs 58, loss:  0.6429 
Epochs 60, loss:  0.6373 
Epochs 62, loss:  0.6308 
Epochs 64, loss:  0.6232 
Epochs 66, loss:  0.6143 
Epochs 68, loss:  0.6039 
Epochs 70, loss:  0.5918 
Epochs 72, loss:  0.5778 
Epochs 74, loss:  0.5618 
Epochs 76, loss:  0.5437 
Epochs 78, loss:

In [8]:
model.eval()
with torch.no_grad():
    logits_test = model(X_test)
    preds =(logits_test.squeeze(1) > 0).long()
    acc =(preds ==y_test).float().mean().item()
    print("Test Accuracy: ",acc)

model.train()

Test Accuracy:  0.8849999904632568


LSTMClassifier(
  (lstm): LSTM(1, 16, batch_first=True)
  (fc): Linear(in_features=16, out_features=1, bias=True)
)